In [ ]:
import pandas as pd
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.decomposition import PCA
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
path = 'C:/Users/rodrigo/Desktop/Machine Learning Aplicada/Nova_atividade/dc_small/'

In [ ]:
def cria_dataframe(caminho):
    arquivos = []
    rotulos = []
    for arquivo in os.listdir(caminho):
        if arquivo.startswith('cat'):
            rotulos.append('cat')
        elif arquivo.startswith('dog'):
            rotulos.append('dog')
        arquivos.append(arquivo)
    return pd.DataFrame({'arquivo': arquivos, 'rotulo': rotulos})

In [ ]:
# Divisão dos dados
df_dados = cria_dataframe(os.path.join(path, 'train'))
train_df, validation_df = train_test_split(df_dados, test_size=0.3, stratify=df_dados['rotulo'], random_state=0)
test_df = cria_dataframe(os.path.join(path, 'test'))

In [ ]:
# Configuração de Data Augmentation
train_datagen = ImageDataGenerator(rotation_range=30, width_shift_range=0.2, height_shift_range=0.2, shear_range=0.2,
                                   zoom_range=0.2, horizontal_flip=True, rescale=1./255)
validation_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
# Função para criar geradores
def create_generator(datagen, dataframe, directory):
    return datagen.flow_from_dataframe(dataframe, directory=directory, x_col='arquivo', y_col='rotulo',
                                       target_size=(224, 224), batch_size=32, class_mode='categorical')

train_generator = create_generator(train_datagen, train_df, os.path.join(path, 'train'))
validation_generator = create_generator(validation_datagen, validation_df, os.path.join(path, 'train'))
test_generator = create_generator(test_datagen, test_df, os.path.join(path, 'test'))


In [ ]:
# Modelo EfficientNetB0 com camadas adicionais
efficientnet_base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
for layer in efficientnet_base.layers:
    layer.trainable = False

efficientnet_model = tf.keras.Sequential([
    efficientnet_base,
    GlobalAveragePooling2D(),
    Dense(1024, activation='relu'),
    Dense(len(train_generator.class_indices), activation='softmax')
])

In [ ]:
# Compilação com taxa de aprendizado ajustada
optimizer = Adam(learning_rate=0.0001)  # Taxa de aprendizado menor
efficientnet_model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
# Callbacks: EarlyStopping e ReduceLROnPlateau
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=0.00001)

In [ ]:
efficientnet_model.fit(train_generator, validation_data=validation_generator, epochs=25)

In [ ]:
# Verifique os rótulos das classes no modelo
print(train_generator.class_indices)

In [ ]:
# Salvar o modelo treinado
efficientnet_model.save('efficientnet_model_improved.keras')

In [ ]:
imagem = path + '10.jpg'

In [ ]:
path

In [ ]:
# Função PCA
def apply_pca_to_images(generator, n_components=32):
    # Obtendo as imagens e rótulos
    images, labels = next(generator)
    
    # Convertendo imagens para vetores 1D
    images_reshaped = images.reshape(images.shape[0], -1)
    
    # Normalização
    scaler = StandardScaler()
    images_scaled = scaler.fit_transform(images_reshaped)
    
    # Aplicando PCA
    pca = PCA(n_components=n_components)
    images_pca = pca.fit_transform(images_scaled)
    
    return images_pca, labels

In [ ]:
# Aplicando PCA nas imagens de treino, validação e teste
train_images_pca, train_labels_pca = apply_pca_to_images(train_generator)
validation_images_pca, validation_labels_pca = apply_pca_to_images(validation_generator)
test_images_pca, test_labels_pca = apply_pca_to_images(test_generator)

In [ ]:
# Exibindo PCA
print("PCA aplicado nas imagens.")

# Função para classificar uma imagem individual

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img
from tensorflow.keras.preprocessing import image  # Corrigido
from tensorflow.keras.applications.efficientnet import EfficientNetB0, preprocess_input
from tensorflow.keras.models import load_model
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

In [ ]:
# Defina o caminho para os dados
path = 'C:/Users/rodrigo/Desktop/Machine Learning Aplicada/Nova_atividade/dc_small/'

# Carregar o modelo treinado (EfficientNetB0)
modelo = load_model('efficientnet_model_improved.keras')  # Substitua com o nome correto do seu modelo salvo

# Defina as classes (labels) do seu modelo
class_labels = ['cat', 'dog']  # Substitua pelos seus rótulos reais, se necessário

In [ ]:
def classifica_imagem(imagem):
    """
    Função para carregar, pré-processar e classificar uma imagem usando o modelo carregado.
    """
    # Carregar a imagem e redimensioná-la para o tamanho esperado pelo modelo (224x224)
    img = load_img(imagem, target_size=(224, 224))  # Tamanho padrão para EfficientNetB0
    
    # Converter a imagem para um array NumPy
    img_ = image.img_to_array(img)  # Usando a função correta
    img_ = np.expand_dims(img_, axis=0)  # Expandir a dimensão para representar o batch
    
    # Pré-processamento da imagem (necessário para EfficientNet)
    img_ = preprocess_input(img_)
    
    # Predição usando o modelo carregado
    preds = modelo.predict(img_)
    
    # Mostrar as probabilidades para cada classe
    for i, label in enumerate(class_labels):
        print(f"Probabilidade para {label}: {preds[0][i]:.4f}")

    # Encontrar a classe predita com a maior probabilidade
    classe_predita = np.argmax(preds, axis=1)[0]
    rotulo_predito = class_labels[classe_predita]
    
    # Exibir o resultado da predição
    print(f"Classe predita: {rotulo_predito} - Confiança: {preds[0][classe_predita]:.4f}")
    
    # Exibir a imagem com o título da classe predita
    plt.imshow(img)
    plt.title(f"Classe: {rotulo_predito}")
    plt.axis("off")  # Desabilitar os eixos da imagem
    plt.show()

In [ ]:
classifica_imagem(imagem)

In [ ]:
# Verifique as classes e suas correspondências
print(train_generator.class_indices)

# Caminho da imagem para predição

In [ ]:
# Matriz de Confusão
predicoes_efficientnet = efficientnet_model.predict(test_generator, steps=test_generator.samples // test_generator.batch_size)
y_pred = np.argmax(predicoes_efficientnet, axis=1)
y_true = test_generator.classes[:len(y_pred)]

In [ ]:
conf_matrix = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=class_labels, yticklabels=class_labels)
plt.title('Matriz de Confusão - EfficientNetB0')
plt.xlabel('Predição')
plt.ylabel('Real')
plt.show()

In [ ]:
conf_matrix = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=class_labels, yticklabels=class_labels)
plt.title('Matriz de Confusão - EfficientNetB0')
plt.xlabel('Predição')
plt.ylabel('Real')
plt.show()

In [ ]:
from sklearn.metrics import classification_report

In [ ]:
# Relatório de Classificação
print("Relatório de Classificação - EfficientNetB0:")
print(classification_report(y_true, y_pred, target_names=class_labels))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Exemplo de resultados fictícios para demonstrar o funcionamento
class_labels = ['cat', 'dog']  # Labels das classes

results = {
    "Logistic Regression": {
        "Classification Report": classification_report(
            [0, 1, 0, 1],  # Rótulos verdadeiros (exemplo)
            [0, 1, 1, 1],  # Rótulos preditos (exemplo)
            target_names=class_labels
        ),
        "Confusion Matrix": confusion_matrix(
            [0, 1, 0, 1],  # Rótulos verdadeiros
            [0, 1, 1, 1]   # Rótulos preditos
        )
    },
    "EfficientNet": {
        "Classification Report": classification_report(
            [0, 1, 0, 1],  # Rótulos verdadeiros
            [0, 0, 1, 1],  # Rótulos preditos
            target_names=class_labels
        ),
        "Confusion Matrix": confusion_matrix(
            [0, 1, 0, 1],  # Rótulos verdadeiros
            [0, 0, 1, 1]   # Rótulos preditos
        )
    }
}

# Iterar sobre os resultados para exibir os relatórios e as matrizes de confusão
for name, result in results.items():
    print(f"\n{name} Results:")
    print(result["Classification Report"])
    
    # Plotar a matriz de confusão
    plt.figure(figsize=(6, 4))
    sns.heatmap(result["Confusion Matrix"], annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_labels, yticklabels=class_labels)
    plt.title(f"Confusion Matrix - {name}")
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()
